# Técnicas de Diseño y Validación de Modelos · Universidad Blas Pascal
## Módulo 2 · Validación y Evaluación de Modelos
## Laboratorio: ROC vs. Precision-Recall en clases desbalanceadas

<div style="display:flex; justify-content:flex-start; margin: 8px 0 18px 0;">
  <a href="https://colab.research.google.com/github/GabrielaOjcius/Laboratorios-Tecnicas-Diseno-Validacion-Modelos/blob/main/Laboratorios%20opcionales/M2_Notebook.ipynb" target="_blank">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/>
  </a>
  <span style="font-size: 0.98em;">En Colab, hac&eacute; una copia en tu Drive antes de editar.</span>
</div>


---

### Objetivo
Demostrar empíricamente por qué el ROC-AUC puede ser engañosamente optimista cuando la clase positiva es muy minoritaria, y por qué la Average Precision (PR-AUC) es la métrica más informativa en esos contextos. El experimento controla una sola variable: el balance de clases.

### Alineación con el material teórico
Este notebook corresponde a la **Actividad** del módulo `Módulo 2 · Validación y Evaluación de Modelos`. Prepara la **Sección C** de la Parte 1 de la Evaluación Integradora.

### Instrucciones
1. Ejecutar las celdas en orden (Entorno de ejecución → Ejecutar todo).
2. Completar las celdas de respuesta marcadas con `# ✏️ RESPONDA AQUÍ`.
3. No modificar el código de las secciones B.1 a B.6 salvo que se indique.

In [ ]:
# ─── B.1. Importaciones y generación de datasets ─────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, auc,
    confusion_matrix, classification_report
)

SEED = 42

def crear_dataset(weights, seed=SEED, n=2000):
    """Genera un dataset con la proporción de clases indicada."""
    X, y = make_classification(
        n_samples=n, n_features=20, n_informative=10,
        n_redundant=5, weights=weights, class_sep=0.9, random_state=seed
    )
    return train_test_split(X, y, test_size=0.20, stratify=y, random_state=seed)

# Dataset 1: clases equilibradas
Xb_tr, Xb_te, yb_tr, yb_te = crear_dataset([0.5, 0.5])

# Dataset 2: desbalanceado — 5 % positivos (típico de fraude, falla, enfermedad rara)
Xd_tr, Xd_te, yd_tr, yd_te = crear_dataset([0.95, 0.05])

print('Dataset balanceado:     train=%d test=%d  | prevalencia train=%.3f test=%.3f'
      % (len(yb_tr), len(yb_te), yb_tr.mean(), yb_te.mean()))
print('Dataset desbalanceado:  train=%d test=%d  | prevalencia train=%.3f test=%.3f'
      % (len(yd_tr), len(yd_te), yd_tr.mean(), yd_te.mean()))

In [ ]:
# ─── B.2. Entrenamiento con Pipeline + CV estratificada ───────────────────────
def construir_pipe():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(max_iter=2000, random_state=SEED))
    ])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for nombre, Xtr, ytr in [('Balanceado', Xb_tr, yb_tr), ('Desbalanceado', Xd_tr, yd_tr)]:
    cv_res = cross_validate(
        construir_pipe(), Xtr, ytr, cv=skf,
        scoring=['roc_auc', 'average_precision']
    )
    print(f'--- {nombre} ---')
    print('  ROC-AUC CV:  %.3f ± %.3f' % (cv_res['test_roc_auc'].mean(), cv_res['test_roc_auc'].std()))
    print('  PR-AUC CV:   %.3f ± %.3f' % (cv_res['test_average_precision'].mean(), cv_res['test_average_precision'].std()))

In [ ]:
# ─── B.3. Evaluación final sobre test set (una sola vez) ─────────────────────
pipe_b = construir_pipe().fit(Xb_tr, yb_tr)
pipe_d = construir_pipe().fit(Xd_tr, yd_tr)

proba_b = pipe_b.predict_proba(Xb_te)[:, 1]
proba_d = pipe_d.predict_proba(Xd_te)[:, 1]

print('═══════════════════════════════════════════════')
print('MÉTRICAS SOBRE TEST SET')
print('═══════════════════════════════════════════════')
for nombre, yte, proba in [('Balanceado',    yb_te, proba_b),
                             ('Desbalanceado', yd_te, proba_d)]:
    pred = (proba >= 0.5).astype(int)
    print(f'\n--- {nombre} (prevalencia={yte.mean():.3f}) ---')
    print(f'  ROC-AUC:  {roc_auc_score(yte, proba):.3f}')
    print(f'  AP:       {average_precision_score(yte, proba):.3f}')
    print(f'  Baseline AP (aleatorio): {yte.mean():.3f}')
    print('  Matriz de confusión:')
    print(confusion_matrix(yte, pred))
    print('  Reporte:')
    print(classification_report(yte, pred, digits=3))

In [ ]:
# ─── B.4. Curvas ROC y Precision-Recall — grilla 2×2 ─────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('ROC vs. Precision-Recall: balanceado vs. desbalanceado', fontsize=13, fontweight='bold')

datasets = [
    (Xb_te, yb_te, proba_b, 'Balanceado (50/50)'),
    (Xd_te, yd_te, proba_d, 'Desbalanceado (95/5)'),
]

for col, (_, yte, proba, titulo) in enumerate(datasets):
    # Fila 0: Curva ROC
    fpr, tpr, _ = roc_curve(yte, proba)
    roc_auc = auc(fpr, tpr)
    axes[0, col].plot(fpr, tpr, lw=2, color='#8B1A2F', label=f'AUC = {roc_auc:.3f}')
    axes[0, col].plot([0, 1], [0, 1], '--', color='gray', label='Aleatorio (AUC=0.5)')
    axes[0, col].set_title(f'Curva ROC — {titulo}', fontsize=11)
    axes[0, col].set_xlabel('FPR (1 − Especificidad)')
    axes[0, col].set_ylabel('TPR (Recall)')
    axes[0, col].legend(); axes[0, col].grid(alpha=0.3)

    # Fila 1: Curva Precision-Recall
    prec, rec, _ = precision_recall_curve(yte, proba)
    ap = average_precision_score(yte, proba)
    axes[1, col].plot(rec, prec, lw=2, color='#8B1A2F', label=f'AP = {ap:.3f}')
    axes[1, col].axhline(yte.mean(), ls='--', color='gray',
                         label=f'Baseline (prevalencia = {yte.mean():.3f})')
    axes[1, col].set_title(f'Curva PR — {titulo}', fontsize=11)
    axes[1, col].set_xlabel('Recall')
    axes[1, col].set_ylabel('Precisión')
    axes[1, col].set_xlim(0, 1); axes[1, col].set_ylim(0, 1)
    axes[1, col].legend(); axes[1, col].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ─── B.5. Intervalo de confianza por bootstrap (dataset desbalanceado) ────────
rng = np.random.default_rng(SEED)
n_test = len(yd_te)
aucs_boot, aps_boot = [], []

for _ in range(1000):
    idx = rng.integers(0, n_test, size=n_test)
    # Saltar muestras con una sola clase (no se puede calcular AUC)
    if len(np.unique(yd_te[idx])) < 2:
        continue
    aucs_boot.append(roc_auc_score(yd_te[idx], proba_d[idx]))
    aps_boot.append(average_precision_score(yd_te[idx], proba_d[idx]))

lo_a, hi_a = np.percentile(aucs_boot, [2.5, 97.5])
lo_p, hi_p = np.percentile(aps_boot,  [2.5, 97.5])

print('Dataset DESBALANCEADO — Intervalos de confianza al 95 % (bootstrap, 1000 remuestreos)')
print(f'  ROC-AUC = {roc_auc_score(yd_te, proba_d):.3f}   IC = [{lo_a:.3f}, {hi_a:.3f}]   ancho = {hi_a-lo_a:.3f}')
print(f'  AP      = {average_precision_score(yd_te, proba_d):.3f}   IC = [{lo_p:.3f}, {hi_p:.3f}]   ancho = {hi_p-lo_p:.3f}')

# Visualización distribuciones bootstrap
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Distribuciones bootstrap — Dataset desbalanceado', fontweight='bold')
for ax, vals, nombre, (lo, hi) in zip(
    axes,
    [aucs_boot, aps_boot],
    ['ROC-AUC', 'Average Precision'],
    [(lo_a, hi_a), (lo_p, hi_p)]
):
    ax.hist(vals, bins=40, color='#8B1A2F', alpha=0.7)
    ax.axvline(lo, color='gray', ls='--', label=f'IC 95%: [{lo:.3f}, {hi:.3f}]')
    ax.axvline(hi, color='gray', ls='--')
    ax.set_title(nombre); ax.set_xlabel(nombre); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ─── B.6. Clasificador trivial: ¿qué AUC y AP tiene? ─────────────────────────
print('=== Clasificador trivial — Dataset desbalanceado ===')
print(f'Prevalencia (clase positiva): {yd_te.mean():.3f}')
print()

# Score aleatorio
rng2 = np.random.default_rng(0)
proba_random = rng2.uniform(0, 1, size=len(yd_te))
print('Score ALEATORIO (uniform[0,1]):')
print(f'  ROC-AUC = {roc_auc_score(yd_te, proba_random):.3f}   (esperado ≈ 0.500)')
print(f'  AP      = {average_precision_score(yd_te, proba_random):.3f}   (esperado ≈ prevalencia = {yd_te.mean():.3f})')
print()

# Score constante = prevalencia
proba_const = np.full(len(yd_te), yd_te.mean())
print('Score CONSTANTE (= prevalencia):')
print(f'  ROC-AUC = {roc_auc_score(yd_te, proba_const):.3f}   (esperado = 0.500)')
print(f'  AP      = {average_precision_score(yd_te, proba_const):.3f}   (esperado = prevalencia = {yd_te.mean():.3f})')
print()
print('Conclusión: el baseline de AP es la prevalencia, no 0.5.')
print('Un modelo útil en datos desbalanceados debe superar sustancialmente la AP baseline.')

---
## Preguntas de análisis
Responder en las siguientes celdas markdown. Hacer referencia explícita a los valores numéricos de la tabla de la sección B.3 y a los gráficos de la sección B.4.

### Pregunta 1
¿Cómo varía el ROC-AUC entre el dataset balanceado y el desbalanceado? ¿Y el Average Precision? ¿Cuál de las dos métricas refleja mejor el efecto del desbalance? Cuantificar la diferencia en ambos casos.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 2
Si la clase positiva representa un evento crítico (fraude bancario, diagnóstico de enfermedad, falla industrial), ¿qué métrica priorizaría para reportar el desempeño del modelo? Justificar en términos de lo que cada métrica mide y del costo relativo de cada tipo de error.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 3
Analizando la curva Precision-Recall del dataset desbalanceado, ¿qué precisión obtiene el modelo para un recall del 80 %? ¿Es aceptable ese nivel de precisión para un sistema de detección de fraude bancario? Razonar en términos del impacto operativo de las falsas alarmas.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 4
Reportar el intervalo de confianza del ROC-AUC y del AP del dataset desbalanceado (sección B.5). ¿Cuál de las dos métricas tiene mayor incertidumbre relativa (ancho del IC / valor central)? ¿Por qué?

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 5
El clasificador trivial tiene ROC-AUC = 0.5 y AP ≈ prevalencia. ¿Confirma o refuta esto la intuición de que un ROC-AUC alto en datos desbalanceados puede ser engañoso? ¿Qué conclusión extrae sobre el poder discriminativo relativo de AUC y AP en este contexto?

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`